# BBQ 데이터셋 → 대회 포맷 변환

HuggingFace의 BBQ 데이터셋을 대회 CSV 포맷으로 변환합니다.

**실행 전 확인사항**
- Kaggle 노트북 우측 사이드바 → Session options → Internet: **ON**
- Accelerator: **None** (GPU 불필요)

**출력 결과**
- `/kaggle/working/bbq_data/train.csv`
- `/kaggle/working/bbq_data/val.csv`
- `/kaggle/working/bbq_data/images/*.jpg` (placeholder 이미지)

In [ ]:
!pip install -q pillow requests

In [ ]:
import json
import requests
from pathlib import Path

import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

## 설정

In [ ]:
OUTPUT_DIR  = Path("/kaggle/working/bbq_data")
MAX_SAMPLES = None   # None = 전체 사용 / 숫자 입력 시 해당 수로 제한 (예: 20000)
VAL_RATIO   = 0.1    # 검증셋 비율
SEED        = 42

BBQ_CATEGORIES = [
    "Age",
    "Disability_status",
    "Gender_identity",
    "Nationality",
    "Physical_appearance",
    "Race_ethnicity",
    "Race_x_gender",
    "Race_x_SES",
    "Religion",
    "SES",
    "Sexual_orientation",
]

IMAGES_DIR = OUTPUT_DIR / "images"
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

print(f"출력 경로: {OUTPUT_DIR}")
print(f"카테고리 수: {len(BBQ_CATEGORIES)}")

## BBQ 데이터 다운로드 (GitHub 레포 zip)

In [ ]:
import io
import zipfile

# GitHub 레포 전체를 zip으로 한 번에 다운로드 (개별 파일 요청보다 안정적)
ZIP_URL  = "https://github.com/nyu-mll/BBQ/archive/refs/heads/main.zip"
ZIP_PATH = Path("/tmp/bbq_repo.zip")
REPO_DIR = Path("/tmp/bbq_repo")

print("BBQ 레포 다운로드 중...")
resp = requests.get(ZIP_URL, timeout=120)
resp.raise_for_status()
ZIP_PATH.write_bytes(resp.content)
print(f"다운로드 완료: {len(resp.content) / 1e6:.1f} MB")

print("압축 해제 중...")
with zipfile.ZipFile(ZIP_PATH) as zf:
    zf.extractall(REPO_DIR)

DATA_DIR = REPO_DIR / "BBQ-main" / "data"
jsonl_files = sorted(DATA_DIR.glob("*.jsonl"))
print(f"발견된 JSONL 파일: {len(jsonl_files)}개")
for f in jsonl_files:
    print(f"  {f.name}")

## BBQ JSONL 파싱

In [ ]:
all_rows = []

for jsonl_path in tqdm(jsonl_files, desc="파싱 중"):
    category = jsonl_path.stem  # 파일명 = 카테고리명

    with open(jsonl_path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            ex = json.loads(line)
            all_rows.append({
                "context":           ex["context"],
                "question":          ex["question"],
                "ans0":              ex["ans0"],
                "ans1":              ex["ans1"],
                "ans2":              ex["ans2"],
                "label":             int(ex["label"]),
                "category":          category,
                "context_condition": ex.get("context_condition", ""),
                "question_polarity": ex.get("question_polarity", ""),
            })

df_raw = pd.DataFrame(all_rows)
print(f"\n총 로드된 샘플 수: {len(df_raw):,}")
print(df_raw["context_condition"].value_counts())

if MAX_SAMPLES is not None:
    df_raw = (
        df_raw
        .groupby("context_condition", group_keys=False)
        .apply(lambda x: x.sample(
            n=min(len(x), MAX_SAMPLES // 2),
            random_state=SEED,
        ))
        .reset_index(drop=True)
    )
    print(f"\n샘플링 후: {len(df_raw):,}")
else:
    print(f"\n전체 사용: {len(df_raw):,}")

## 대회 포맷 변환 + Placeholder 이미지 생성

In [ ]:
records = []

for idx, row in tqdm(df_raw.iterrows(), total=len(df_raw), desc="변환 중"):
    img_name = f"bbq_img_{idx:06d}.jpg"
    Image.new("RGB", (224, 224), color=(128, 128, 128)).save(IMAGES_DIR / img_name)

    records.append({
        "sample_id":         f"BBQ_{idx:06d}",
        "image_path":        f"./images/{img_name}",
        "context":           row["context"],
        "question":          row["question"],
        "answers":           json.dumps([row["ans0"], row["ans1"], row["ans2"]]),
        "label":             row["label"],
        "category":          row["category"],
        "context_condition": row["context_condition"],
        "question_polarity": row["question_polarity"],
    })

df = pd.DataFrame(records)
print(f"변환 완료: {len(df):,}개")

## Train / Val 분리 및 저장

In [ ]:
val_df   = df.sample(frac=VAL_RATIO, random_state=SEED)
train_df = df.drop(val_df.index).reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

train_df.to_csv(OUTPUT_DIR / "train.csv", index=False)
val_df.to_csv(OUTPUT_DIR / "val.csv",   index=False)

print(f"Train: {len(train_df):,}개 → {OUTPUT_DIR / 'train.csv'}")
print(f"Val  : {len(val_df):,}개 → {OUTPUT_DIR / 'val.csv'}")

## 결과 확인

In [ ]:
print("="*50)
print(f"총 샘플 수  : {len(df):,}")
print(f"Train       : {len(train_df):,}")
print(f"Val         : {len(val_df):,}")
print(f"\n[label 분포]")
print(df["label"].value_counts().to_string())
print(f"\n[context_condition 분포]")
print(df["context_condition"].value_counts().to_string())
print(f"\n[카테고리 분포]")
print(df["category"].value_counts().to_string())
print("="*50)
print("\n샘플 미리보기:")
train_df[["sample_id", "context", "question", "answers", "label", "context_condition"]].head(3)

## 완료

노트북 우측 **Output** 탭에서 `bbq_data/` 폴더를 확인하고
**New Dataset** 버튼으로 저장하면 파인튜닝 노트북에서 재사용할 수 있습니다.